<a href="https://colab.research.google.com/github/karin-kaito/202604-llm/blob/main/ft_1_%E5%88%86%E9%A1%9E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 分類のためのファインチューニング

In [ ]:
%%python
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

# --- 1. トークナイザーとモデルのロード ---
# トークナイザーの初期化: 指定されたモデルの事前学習済みトークナイザーをロードします。
# `use_fast=True`で高速なRustベースのトークナイザーを使用します。
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", use_fast=True)

# モデルの初期化: 指定されたモデルの事前学習済みモデルをシーケンス分類用にロードします。
# `num_labels=2`で2クラス分類（例: ポジティブ/ネガティブ）を設定します。
# `torch_dtype="auto"`で利用可能な最適なデータ型（例: fp16）を自動選択します。
# `trust_remote_code=True`で、リモート（Hugging Face Hub）からカスタムコードをロードすることを許可します。
model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    num_labels=2,
    torch_dtype="auto",
    trust_remote_code=True,
)

# モデルのパディングトークンIDを設定します。これは、バッチ処理時にシーケンス長を揃えるために必要です。
model.config.pad_token_id = tokenizer.pad_token_id

# --- 2. データセットの準備 ---
# サンプルのデータセットをPython辞書から作成します。
# `text`は入力テキスト、`labels`は対応する分類ラベル（0:ネガティブ、1:ポジティブ）です。
ds = Dataset.from_dict(
    {
        "text": [
            "I love this movie. Fantastic!",
            "Terrible experience... never again.",
            "It was okay, not great.",
            "Brilliant acting and strong plot.",
        ],
        "labels": [1, 0, 0, 1],
    }
)

# --- 3. データの前処理 ---
# データセットの各例（テキスト）をトークン化し、モデルの入力形式に変換する関数を定義します。
# `truncation=True`で最大長を超えるテキストを切り捨て、`max_length=128`で最大シーケンス長を設定します。
def preprocess(example):
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
    )
    return tokenized

# データセット全体に前処理関数を適用します。元の`text`列は削除されます。
tok_ds = ds.map(preprocess, remove_columns=["text"])

# Data Collatorの初期化: トークン化されたデータをバッチにまとめる際に、動的パディングを適用します。
# `return_tensors="pt"`でPyTorchテンソルを返します。
collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    return_tensors="pt",
)

# --- 4. トレーニング設定 ---
# トレーニングの引数を設定します。
# `output_dir`: モデルのチェックポイントやログを保存するディレクトリ。
# `learning_rate`: 学習率。
# `num_train_epochs`: トレーニングエポック数（データセット全体を何回繰り返すか）。
# `per_device_train_batch_size`: 各デバイスごとのトレーニングバッチサイズ。
# `logging_steps`: 何ステップごとにログを記録するか。
# `report_to="none"`: ログを外部サービスに報告しない設定。
training_args = TrainingArguments(
    output_dir="./ckpt-cls",
    learning_rate=1e-6,
    num_train_epochs=5, # エポック数を1から5に増加
    per_device_train_batch_size=4, # バッチサイズを2から4に増加
    logging_steps=1,
    report_to="none"
)

# --- 5. トレーナーの初期化とトレーニングの実行 ---
# `Trainer`クラスを初期化します。モデル、引数、トレーニングデータセット、データコレーターを渡します。
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_ds,
    data_collator=collator,
)

# トレーニングを開始します。
trainer.train()

## ファインチューニング前後のモデル比較

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- 1. トークナイザーとモデルのロード ---
# トークナイザーのロード（ファインチューニング前と同じものを使用）
tokenizer_compare = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", use_fast=True)

# ファインチューニング前のオリジナルモデルをロード
original_model = AutoModelForSequenceClassification.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    num_labels=2,
    torch_dtype="auto",
    trust_remote_code=True,
)
original_model.config.pad_token_id = tokenizer_compare.pad_token_id

# ファインチューニング後のモデルをロード
# 保存されたチェックポイントのパスを指定します。`./ckpt-cls/checkpoint-2`は前回の実行で保存されたモデルのディレクトリです。
fined_tuned_model = AutoModelForSequenceClassification.from_pretrained(
    "./ckpt-cls/checkpoint-2", # パスを修正
    num_labels=2,
    torch_dtype="auto",
    trust_remote_code=True,
)
fined_tuned_model.config.pad_token_id = tokenizer_compare.pad_token_id

# --- 2. 比較用テキストの準備 ---
# モデルの予測を比較するためのサンプルテキストリスト。
texts_to_compare = [
    "This is a great product!",
    "I hate this movie, it's terrible.",
    "It was an average experience."
]

# --- 3. 予測関数の定義 ---
# 与えられたモデルとテキストで予測を行い、予測されたクラスと各クラスの確率を返すヘルパー関数。
def get_prediction(model, text):
    # テキストをトークン化し、PyTorchテンソル形式でモデル入力を作成。
    inputs = tokenizer_compare(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad(): # 勾配計算を無効にし、メモリ使用量と計算時間を削減。
        logits = model(**inputs).logits # モデルからロジット（生スコア）を取得。
    probabilities = torch.softmax(logits, dim=1) # ロジットをソフトマックス関数で確率に変換。
    predicted_class = torch.argmax(probabilities, dim=1).item() # 最も高い確率を持つクラスIDを取得。
    return predicted_class, probabilities.tolist()[0] # 予測クラスと確率リストを返す。

# --- 4. モデル比較の実行と結果表示 ---
print("--- Model Comparison (0: Negative, 1: Positive) ---")
for text in texts_to_compare:
    # オリジナルモデルとファインチューニング後モデルでそれぞれ予測を実行。
    original_pred_class, original_probs = get_prediction(original_model, text)
    finetuned_pred_class, finetuned_probs = get_prediction(fined_tuned_model, text)

    print(f"\nText: '{text}'")
    # 結果を表示。予測クラスと、そのクラスの確率（小数点以下4桁）を出力。
    print(f"  Original Model: Class {original_pred_class}, Probs: {original_probs[original_pred_class]:.4f}")
    print(f"  Fine-tuned Model: Class {finetuned_pred_class}, Probs: {finetuned_probs[finetuned_pred_class]:.4f}")

In [ ]:
!ls -la ./ckpt-cls/

In [ ]:
!ls -la ./ckpt-cls/*